# 第四版完整运行入口（推荐）

停止原服务器并关闭其内核后，本文件选择新的Python 3内核。逐个运行下面3个代码框。第三个代码框会完成校验与测速，自动启动选中的FPGA点餐路径；正常停止后自动提供日志下载链接。第三版及原硬件不覆盖。

本轮达标与否以输出的V4_GOALS为准，不把吞吐提升写成延迟下降。新路径收益不稳健时明确保留v3_reference。


## 1．路径与离线运行库

In [ ]:
from pathlib import Path
import os,sys
ROOT=Path('/home/xilinx/jupyter_notebooks')
APP=ROOT/'gesture_order_accel_v4'
for name in ('board_server','order_core','session_guard','setup_runtime','v3_runtime','v4_runtime','pipeline_runtime','validate_v4','start_v4'):
    module=sys.modules.get(name)
    if module is not None and Path(module.__file__).resolve()!=(APP/(name+'.py')).resolve():
        raise RuntimeError('当前内核缓存了其他版本的 '+name+'，请重启本文件内核再从第1部分运行。')
sys.path.insert(0,str(APP));os.chdir(APP)
from setup_runtime import prepare
prepare(ROOT)


## 2．环境与硬件文件检查（本步不下载bitstream）

In [ ]:
from start_v4 import preflight
preflight(ROOT)


## 3．完整板测 → 自动启动 → 停止后自动导出

前两步通过后运行本框。使用包内51张图、14种配置、正反顺序28组测速；每组2个预热结果＋12个测量结果。预留约15—30分钟，输出持续更新时等待，不点运行全部或重复启动。

看到ORDER_BOARD_SERVER_READY port=8765后，板端持续[*]正常；到笔记本运行原start_laptop.bat。完成三菜各一份33元模拟订单及V返回/取消、点赞保持只加一次、移手后减一份、断线恢复回归。

完成后笔记本Ctrl+C，板端■一次，等待ORDER_BOARD_SERVER_STOPPED和下载链接。把板端ZIP及笔记本本次logs发回。若校验报错，本框也会尝试导出已完成证据；线程没退出时会拒绝打包并显示原因，不重新加载硬件。

后续重启内核后，无需重新长测：运行本文件前两步，然后到00_final_candidate_v4.ipynb的第6部分读取最近一次通过报告启动。实际点餐尚未确认前，仍是最终候选。


In [ ]:
from run_complete import run
v4_report=run(ROOT,samples=12)
